# Setup

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from upstash_vector import Index
from google import genai as google_genai
import time

# Ingest

In [ ]:
# carrega variáveis de ambiente
load_dotenv()

UPSTASH_ENDPOINT = os.getenv("UPSTASH_ENDPOINT")
UPSTASH_WRITE_API_KEY = os.getenv("UPSTASH_WRITE_API_KEY")
GEMINI_API_KEY_T1 = os.getenv("GEMINI_API_KEY_T1")
REINGEST = False

# configura cliente Upstash e google genai
index = Index(url=UPSTASH_ENDPOINT, token=UPSTASH_WRITE_API_KEY)
google_client = google_genai.Client(api_key=GEMINI_API_KEY_T1)

print("Conexões configuradas.")

In [3]:
# carrega chunks
with open("../data/chunks/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Chunks carregados: {len(chunks)}")

Chunks carregados: 5783


In [ ]:
# busca source_urls já inseridos no Upstash
def get_existing_urls():
    existing = set()
    cursor = ""

    while True:
        res = index.range(cursor=cursor, limit=1000, include_metadata=True)
        for v in res.vectors:
            existing.add(v.metadata["source_url"])
        cursor = res.next_cursor
        if cursor == "":
            break

    return existing

# ingestão de chunks no upstash com embedding via Gemini
def ingest_chunks(chunks, batch_size=100, start_id=0):
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]
        
        # embeda todos de uma vez com retry
        for attempt in range(5):
            try:
                result = google_client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=texts
                )
                break
            except Exception as e:
                if "429" in str(e):
                    wait = 30 * (attempt + 1)
                    print(f"  Rate limit, aguardando {wait}s...")
                    time.sleep(wait)
                else:
                    raise e
        
        vectors = []
        for j, (chunk, embedding) in enumerate(zip(batch, result.embeddings)):
            vectors.append((
                str(start_id + i + j),
                embedding.values,
                {
                    "text": chunk["text"],
                    "source_url": chunk["source_url"],
                    "title": chunk["title"],
                    "type": chunk["type"],
                    "published_at": chunk.get("published_at")
                }
            ))
        
        index.upsert(vectors=vectors)
        print(f"Inseridos {min(i + batch_size, total)}/{total} chunks")

In [ ]:
# lógica de reingest: 
# Se REINGEST for True, zera o index e reinsere tudo.
# Se False, busca URLs já existentes e só insere os novos chunks.
if REINGEST:
    index.reset()
    print("Index zerado.")
    start_id = 0
    new_chunks = chunks
else:
    existing_urls = get_existing_urls()
    new_chunks = [c for c in chunks if c["source_url"] not in existing_urls]
    start_id = index.info().vector_count
    print(f"Chunks novos a inserir: {len(new_chunks)}")

ingest_chunks(new_chunks, start_id=start_id)
print("Ingestão concluída.")

In [ ]:
url_alvo = "https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view"

results = index.range(
    cursor="",
    limit=10,
    include_metadata=True,
    filter=f"source_url = '{url_alvo}'"
)

print(f"Chunks no Upstash: {len(results.vectors)}")
for v in results.vectors:
    print(f"\n{v.metadata['text'][:]}")
    print("---")